In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
import random
import pandas as pd
from urllib.parse import urljoin, urlparse
from tqdm import tqdm

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
}

def fetch(url, timeout=10, max_retries=3):
    """Download halaman HTML dengan retry sederhana."""
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            r.raise_for_status()
            return r.text
        except Exception:
            time.sleep(1 + attempt)
    raise RuntimeError(f"Gagal mengambil halaman: {url}")

In [ ]:
def parse_article(html, url):
    """Parse 1 artikel Detik.com menjadi dict."""
    soup = BeautifulSoup(html, "lxml")

    # Judul
    title = ""
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(strip=True)

    # Tanggal
    date_text = ""
    time_tags = soup.find_all("time")
    if time_tags:
        date_text = time_tags[0].get_text(" ", strip=True)

    if not date_text:
        text = soup.get_text(" ", strip=True)
        m = re.search(r'\d{1,2}\s\w+\s\d{4}', text)
        if m:
            date_text = m.group(0)

    # Isi berita
    paragraphs = []
    body_candidates = soup.select("div.detail__body-text") \
                       + soup.select("div.detail_text") \
                       + soup.select("div.itp_bodycontent")

    if body_candidates:
        for p in body_candidates[0].find_all("p"):
            txt = p.get_text(" ", strip=True)
            if txt:
                paragraphs.append(txt)

    if not paragraphs:
        for p in soup.find_all("p"):
            txt = p.get_text(" ", strip=True)
            if txt:
                paragraphs.append(txt)

    content = "\n\n".join(paragraphs)

    return {
        "Judul_berita": title,
        "Isi": content,
        "Date": date_text,
        "Url": url
    }

In [ ]:
def extract_links_from_tagpage(html, base_url):
    soup = BeautifulSoup(html, "lxml")
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if re.search(r'/d-\d+/', href) or '/berita-' in href:
            if href.startswith("//"):
                href = "https:" + href
            elif href.startswith("/"):
                href = urljoin(base_url, href)

            parsed = urlparse(href)
            if parsed.scheme in ("http", "https"):
                links.add(href.split("?")[0])

    return list(links)

In [ ]:
def crawl_tag(tag_url, max_articles=50):
    html = fetch(tag_url)
    links = extract_links_from_tagpage(html, tag_url)

    results = []

    for link in tqdm(links[:max_articles], desc="Mengambil artikel"):
        try:
            art_html = fetch(link)
            data = parse_article(art_html, link)
            results.append(data)
            time.sleep(random.uniform(1, 3))
        except Exception as e:
            print("Gagal:", link, e)

    return results

In [ ]:
TAG_URL = "https://www.detik.com/tag/ekonomi"

articles = crawl_tag(TAG_URL, max_articles=50)
len(articles)

In [ ]:
df = pd.DataFrame(articles)
df.to_csv("raw.csv", index=False, encoding="utf-8-sig")
df.head()
